# Level 3 — Core Numerical Methods Engine

## Scientific Objectives
1. Implement Root Finding (Bisection, Newton-Raphson, Secant) to calculate precise irrigation deficits considering non-linear drainage.
2. Implement Finite Differences (Forward, Backward, Central) for rate of moisture change.
3. Implement Numerical Integration (Trapezoidal, Simpson's) for cumulative deficits.
4. Implement Gaussian Elimination for multi-zone water allocation.

In [ ]:
import numpy as np
import pandas as pd

# Import our modular numerical methods!
from src.numerical_methods import (
    bisection, newton_raphson, secant, 
    forward_diff, backward_diff, central_diff, 
    trapezoidal_rule, simpson_rule, gaussian_elimination
)

# 1. ROOT FINDING: Solving for required irrigation (I)
# Target soil = S_t + R - ET - D(I) + I
def f_irrigation(I, S_t, target, R, ET, FC, coeff):
    drainage = coeff * max(0, S_t + I - FC)
    return (S_t + R - ET - drainage + I) - target

# Derivative of the irrigation function (Required for Newton-Raphson)
def df_irrigation(I, S_t, FC, coeff):
    # If water exceeds field capacity, drainage triggers, changing the rate of accumulation
    if S_t + I > FC:
        return 1.0 - coeff
    return 1.0

# Example test scenario
S_t, target, R, ET, FC, coeff = 20.0, 35.0, 0.0, 5.0, 41.0, 0.18

# Create lambda functions bound to our specific daily variables
f_val = lambda I: f_irrigation(I, S_t, target, R, ET, FC, coeff)
df_val = lambda I: df_irrigation(I, S_t, FC, coeff)

# Run all three methods
root_bisect, i_bisect = bisection(f_val, 0, 50)
root_newton, i_newton = newton_raphson(f_val, df_val, x0=10)
root_secant, i_secant = secant(f_val, 0, 50)

In [ ]:
# Create the required Convergence Comparison Table
comparison_data = {
    "Method": ["Bisection", "Newton-Raphson", "Secant"],
    "Calculated Root (mm)": [root_bisect, root_newton, root_secant],
    "Iterations Required": [i_bisect, i_newton, i_secant],
    "Convergence Status": ["Success", "Success", "Success"]
}

df_convergence = pd.DataFrame(comparison_data)
print("=== ROOT FINDING CONVERGENCE COMPARISON ===")
display(df_convergence)

In [ ]:
# 2. NUMERICAL DIFFERENTIATION
# Estimating rate of moisture change (dS/dt) over 5 days
moisture = np.array([33.2, 36.1, 33.7, 31.8, 33.3]) 
dt = 1.0

fwd = forward_diff(moisture, dt)
bwd = backward_diff(moisture, dt)
cen = central_diff(moisture, dt)

print("\n=== FINITE DIFFERENCES (Rate of Moisture Change) ===")
df_diff = pd.DataFrame({
    "Day": [1, 2, 3, 4, 5],
    "Moisture (%)": moisture,
    "Forward dS/dt": fwd,
    "Backward dS/dt": bwd,
    "Central dS/dt": cen
})
display(df_diff.round(2))

In [ ]:
# 3. NUMERICAL INTEGRATION
# Sample ET array over an even number of intervals (odd length array)
et_sample = np.array([4.1, 4.2, 4.5, 4.6, 4.4, 4.1, 3.8]) 
dx = 1.0

trap_res = trapezoidal_rule(et_sample, dx)
simp_res = simpson_rule(et_sample, dx)

print("\n=== NUMERICAL INTEGRATION (Cumulative ET Deficit) ===")
print(f"Trapezoidal Rule: {trap_res:.2f} mm")
print(f"Simpson's Rule:   {simp_res:.2f} mm")
print(f"Difference:       {abs(trap_res - simp_res):.4f} mm")

In [ ]:
# 4. LINEAR SYSTEMS (Gaussian Elimination)
# Suppose we have 3 zones competing for a fixed 2000L water tank.
# Constraint 1: ZoneA + ZoneB + ZoneC = 2000
# Constraint 2: ZoneA needs twice as much as ZoneB: A - 2B = 0
# Constraint 3: ZoneC needs 1.5x of ZoneB: C - 1.5B = 0

A = np.array([
    [1.0,  1.0, 1.0], 
    [1.0, -2.0, 0.0], 
    [0.0, -1.5, 1.0]
])
b = np.array([2000.0, 0.0, 0.0])

# Call our modular solver
allocation = gaussian_elimination(A, b)

print("\n=== GAUSSIAN ELIMINATION: WATER ALLOCATION ===")
print(f"Zone A Allocation: {allocation[0]:.1f} Liters")
print(f"Zone B Allocation: {allocation[1]:.1f} Liters")
print(f"Zone C Allocation: {allocation[2]:.1f} Liters")

# Verification
print(f"Total Allocated:   {sum(allocation):.1f} Liters (Target: 2000.0)")